In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/movie-recommendation-hackathon-2026/sample_submission.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/movies.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/imdb_data.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_tags.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_scores.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/train.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/test.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/tags.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/links.csv


In [47]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

In [3]:
train_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/train.csv')

train_df.head()

,userId,movieId,rating,timestamp
0,5163,57669,4.0,1518349992
1,106343,5,4.5,1206238739
2,146790,5459,5.0,1076215539
3,106362,32296,2.0,1423042565
4,9041,366,3.0,833375837


In [4]:
movies_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/movies.csv')

movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
# Encode movie genres with MultiLabelBinarizer
genres = movies_df['genres'].str.split('|')

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(genres)

def content_based_recommender(train_df, movies_df, genre_matrix, user_id, n=10):
    # Filter train_df to obtain user's previously rated movies
    user_movies = train_df.loc[
        train_df['userId'] == user_id, ['movieId', 'rating']
    ]

    # Look up the corresponding movies in movies_df
    user_movie_details = user_movies.merge(movies_df, on='movieId', how='inner')

    genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=movies_df.index)

    liked_positions = np.flatnonzero(
        movies_df['movieId'].isin(user_movie_details['movieId']).to_numpy()
    )

    user_genres = genre_matrix[liked_positions]

    watched_mask = movies_df['movieId'].isin(user_movies['movieId']).to_numpy()

    watched_genres = genre_matrix[watched_mask]

    similarity_matrix = cosine_similarity(
        genre_matrix,
        watched_genres
    )

    ratings_map = user_movies.set_index('movieId')['rating'].to_dict()

    watched_ratings = movies_df.loc[
        watched_mask, 'movieId'
    ].map(ratings_map).to_numpy()

    # Convert ratings into preference weights
    rating_weights = watched_ratings - 3

    # Calculate recommendation scores
    weighted_scores = similarity_matrix @ rating_weights
    similarity_totals = similarity_matrix.sum(axis=1)

    scores = np.divide(
        weighted_scores,
        similarity_totals,
        out=np.zeros_like(weighted_scores, dtype=float),
        where=similarity_totals != 0
    )

    # # Exclude already watched movies
    # watched_movies_ids = user_movies['movieId'].unique()
    # unwatched_mask = ~movies_df['movieId'].isin(watched_movies_ids)

    # # Create a recommendations DataFrame
    # recommendations = movies_df.loc[
    #     unwatched_mask, ['movieId', 'title']
    # ].copy()

    # recommendations['score'] = scores[unwatched_mask.to_numpy()]

    # recommendations = recommendations[
    #     similarity_totals[unwatched_mask.to_numpy()] > 0
    # ]

    # recommendations = recommendations.sort_values(
    #     by='score',
    #     ascending=False
    # ).head(n)

    predicted_ratings = np.clip(scores + 3, 0.5, 5.0)

    predictions = movies_df[['movieId', 'title']].copy()
    predictions['predicted_rating'] = predicted_ratings

    return predictions
    # return recommendations

In [6]:
content_based_recommender(train_df, movies_df, genre_matrix, 1)

,movieId,title,predicted_rating
0,1,Toy Story (1995),3.850602
1,2,Jumanji (1995),4.152482
2,3,Grumpier Old Men (1995),3.797268
3,4,Waiting to Exhale (1995),3.875793
4,5,Father of the Bride Part II (1995),3.740850
...,...,...,...
62418,209157,We (2018),3.933264
62419,209159,Window of the Soul (2001),3.000000
62420,209163,Bad Poems (2018),3.875396
62421,209169,A Girl Thing (2001),3.000000


In [7]:
# # Encode movie genres with MultiLabelBinarizer
# genres = movies_df['genres'].str.split('|')
# mlb = MultiLabelBinarizer()
# genre_matrix = mlb.fit_transform(genres)

# similarity_matrix = cosine_similarity(genre_matrix)

# def content_based_recommender(
#     train_df,
#     movies_df,
#     similairty_matrix,
#     user_id,
#     n=10
# ):
#     # Filter train_df to obtain user's previously rated movies
#     user_movies = train_df.loc[
#     train_df['userId'] == user_id, ['movieId', 'rating']
#     ]

#     watched_mask = movies_df['movieId'].isin(
#         user_movies['movieId']
#     ).to_numpy()

#     watched_ratings = movies_df.loc[
#     watched_mask, 'movieId'
#     ].map(
#         user_movies.set_index('movieId')['rating']
#     ).to_numpy()

#     # Convert ratings into preference weights
#     rating_weights = watched_ratings - 3

#     # Retrieve similarities instead of recalculating them
#     user_similarities = similarity_matrix[:, watched_mask]

#     # Calculate recommendation scores
#     weighted_scores = user_similarities @ rating_weights
#     similarity_totals = user_similarities.sum(axis=1)

#     scores = np.divide(
#         weighted_scores,
#         similarity_totals,
#         out=np.zeros_like(weighted_scores, dtype=float),
#         where=similarity_totals != 0
#     )

#     # Convert recommendation scores into predicted ratings
#     predicted_ratings = np.clip(scores + 3, 0.5, 5.0)

#     # Create a DataFrame containing all movie predictions
#     predictions = movies_df[['movieId', 'title']].copy()

#     predictions['predicted_rating'] = [predicted_ratings]

#     return predictions

In [8]:
content_based_recommender(train_df, movies_df, genre_matrix, 1)

,movieId,title,predicted_rating
0,1,Toy Story (1995),3.850602
1,2,Jumanji (1995),4.152482
2,3,Grumpier Old Men (1995),3.797268
3,4,Waiting to Exhale (1995),3.875793
4,5,Father of the Bride Part II (1995),3.740850
...,...,...,...
62418,209157,We (2018),3.933264
62419,209159,Window of the Soul (2001),3.000000
62420,209163,Bad Poems (2018),3.875396
62421,209169,A Girl Thing (2001),3.000000


In [30]:
train_df_data, validation_df_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

In [37]:
validation_df_data.head(1)

,userId,movieId,rating,timestamp,predicted_rating
5954604,122380,31445,2.5,1159967140,3.533395


In [43]:
validation_df_data = validation_df_data.copy()

global_mean = train_df['rating'].mean()

validation_df_data['predicted_rating'] = global_mean

# Map movie IDs to their corresponding genre matrix positions
movie_ids_to_pos = pd.Series(
	np.arange(len(movies_df)),
	index=movies_df['movieId']
)

# Group training ratings by user beforehand
train_by_user = {
	user_id: group[['movieId', 'rating']]
	for user_id, group in train_df.groupby('userId')
}

for user_id, user_validation in validation_df_data.groupby('userId', sort=False):
    user_train = train_by_user.get(user_id)

    # Handle users without training ratings
    if user_train is None:
        continue

    # Get genre matrix positions for movies in validation data
    candidate_positions = user_validation['movieId'].map(movie_ids_to_pos)
    valid_candidates = candidate_positions.notna().to_numpy()

    # Handle movies missing from movies_df
    if not valid_candidates.any():
        continue

    # Get training movies that exist in movies_df
    known_watched = user_train[
        user_train['movieId'].isin(movie_ids_to_pos.index)
    ]

    if known_watched.empty:
        continue

    watched_positions = known_watched['movieId'].map(
        movie_ids_to_pos
    ).to_numpy(dtype=int)

    # Extract genre representations
    watched_genres = genre_matrix[watched_positions]

    candidate_indices = np.flatnonzero(valid_candidates)

    candidate_genres = genre_matrix[
        candidate_positions.iloc[candidate_indices].to_numpy(dtype=int)
    ]

    # Calculate similarities only for validation movies
    similarities = cosine_similarity(
        candidate_genres,
        watched_genres
    )

    # Calculate rating weights
    rating_weights = known_watched['rating'].to_numpy() - 3
    weighted_scores = similarities @ rating_weights
    similarity_totals = similarities.sum(axis=1)

    scores = np.divide(
        weighted_scores,
        similarity_totals,
        out=np.zeros_like(weighted_scores, dtype=float),
        where=similarity_totals != 0
    )

    predicted_ratings = np.clip(scores + 3, 0.5, 5.0)

    # Assign predictions to validation records
    validation_df_data.loc[
        user_validation.index[candidate_indices],
        'predicted_rating'
    ] = predicted_ratings


validation_df_data.head()

,userId,movieId,rating,timestamp,predicted_rating
5954604,122380,31445,2.5,1159967140,2.920054
1460784,22380,56775,4.0,1343936580,3.813563
6631832,104339,356,2.5,1111529397,3.391199
5396407,64877,6874,4.0,1513800297,4.087011
8150654,63164,2762,5.0,1005315064,4.015988


In [29]:
# validation_df_data = validation_df_data.iloc[:100000].copy()

# global_mean = train_df['rating'].mean()

# validation_df_data['predicted_rating'] = np.nan

# for user_id, user_validation in validation_df_data.groupby('userId'):
#     if user_id not in train_df['userId'].values:
#         validation_df_data.loc[
#             user_validation.index, 'predicted_rating'
#         ] = global_mean
#         continue

#     # Generate predictions for this user
#     predictions = content_based_recommender(
#         train_df,
#         movies_df,
#         genre_matrix,
#         user_id
#     )

#     # Retrive predictions for the movies in the validation dataset
#     predicted_ratings = predictions.set_index(
#         'movieId'
#     )['predicted_rating'].reindex(user_validation['movieId']).to_numpy()

#     # Assign predictions to validation_df
#     validation_df_data.loc[
#         user_validation.index, 'predicted_rating'
#     ] = predicted_ratings

# # Handle any movies missing from movies_df
# validation_df_data['predicted_rating'] = validation_df_data[
#     'predicted_rating'
# ].fillna(global_mean)

# validation_df_data.head()

KeyboardInterrupt: 

In [45]:
validation_df_data.to_csv('content_based_validation_predictions.csv', index=False)

In [46]:
validation_df_data.shape

(2000008, 5)

In [48]:
rmse = root_mean_squared_error(
    validation_df_data['rating'],
    validation_df_data['predicted_rating']
)

rmse

0.8778431389726122

In [50]:
print("Fallback predictions: ", (validation_df_data['predicted_rating'] == global_mean).sum())

Fallback predictions:  0
